# LILA BC

For my project, I need images of wildlife in the Amazon that include animals. I'm using the [LILA BC](https://lila.science/datasets/wcscameratraps) dataset, but since it contains animals from various countries, I filtered the dataset using a JSON file with labels specific to Latin American countries. I also created a script to download the images to my local machine, as I plan to train my model locally. This file represents that work.

## Counting code of countrys

In [ ]:
import json
import pandas as pd
# path for JSON file
json_path = '../wcs_camera_traps.json'

# read JSON file
with open(json_path, 'r') as f:
    data = json.load(f)
# count the number of images and select the country codes'
if 'images' in data:
    num_images = len(data['images'])
    df_images = pd.json_normalize(data['images'])
    codes = df_images['country_code'].unique()
print(f"Number of images: {num_images}")
print(codes) 



In [ ]:
import pandas as pd

json_path = '../wcs_camera_traps.json'

# read JSON file
with open(json_path, 'r') as f:
    data = json.load(f)
# count the number of images and select the country codes'
if 'categories' in data:
    num_categories = len(data['categories'])
    df_categories = pd.json_normalize(data['categories'])
    categories = df_categories['name']

print(f"Numer of categories: {num_categories}")
print(f"Categories: \n{categories}")



## Country of latin American

- **Bolívia** (bol)
- **Equador** (ecu)
- **Peru** (pry)
- **Venezuela** (ven)


In [ ]:
import pandas as pd

def load_progress(csv_name):
    df_names = pd.read_csv(str(csv_name), header=0, sep=';')
    total = len(df_names)
    print("Foram processadas " + str(total) + " imagens")
    return df_names

def filer_dataset(df_names: pd.DataFrame, df_images: pd.DataFrame):
    codes = ['bol', 'ecu', 'pry', 'ven']
    df_filtered = df_images[~df_images['id'].isin(df_names['image_id'])]
    df_filtered[df_filtered['country_code'].isin(codes)]
    df_filtered.filter(df_filtered['corrupt'] == False)
    return df_filtered

### Creating script for downloand images

In [ ]:
import json
import pandas as pd
import requests
import os
import uuid
import time
import csv

# path for JSON file
json_path = '../wcs_camera_traps.json'

# URL for lila dataset
url = 'https://lilawildlife.blob.core.windows.net/lila-wildlife/wcs-unzipped/'
csv_save_name = 'image_load_sem_animal.csv'
total_image = 10_000
base_path = '../imagens/train/sem_animal'

with open(json_path, 'r') as f:
    data = json.load(f)

    if 'images' in data and 'annotations' in data and 'categories' in data:
        categories = pd.json_normalize(data['categories'])
        annotations = pd.json_normalize(data['annotations'])
        not_request_images = pd.json_normalize(data['images'])
        df_dowload_images = pd.DataFrame(['image_name', 'image_id', 'class_name', "country_code", 'count', 'category_id'])

        if(os.path.exists(f'./{csv_save_name}')):
            df_dowload_images = load_progress(f'./{csv_save_name}')
            not_request_images = filer_dataset(df_dowload_images, not_request_images)

        print("Processed images:", len(df_dowload_images))
        print("Not processed images:", len(not_request_images))
        merged_images = pd.merge(not_request_images, annotations, left_on="id", right_on="image_id", how="inner")
        merged_images = pd.merge(merged_images, categories, left_on="category_id", right_on="id", how="inner")
        display(merged_images.head())
        merged_images = merged_images[merged_images["category_id"] == 0]
        # create the folder images
        if not os.path.exists(base_path):
            os.makedirs(base_path)
        for i in range(0, total_image):

            original_filename = merged_images.iloc[i]["file_name"]
            unique_id = merged_images.iloc[i]["id_x"]
            country_code = merged_images.iloc[i]["country_code"]  
            category_id = merged_images.iloc[i]["category_id"]
            count = merged_images.iloc[i]["count_x"]
            class_name = merged_images.iloc[i]["name"]
            
            _, ext = os.path.splitext(original_filename)
            image_url = url + original_filename
            full_local_path = os.path.join(base_path, "{unique_id}{ext}".format(unique_id=unique_id, ext=ext))
            
            # Create control for time request and do request
            start_time = time.time() 
            response = requests.get(image_url, stream=True)
            end_time = time.time()  


            if response.status_code == 200:
                # Create subfloders
                os.makedirs(os.path.dirname(full_local_path), exist_ok=True)

                # Write the image to a file
                with open(full_local_path, 'wb') as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)
                    # Adiciona ao array (para montar o DataFrame depois)

                write_header = not os.path.exists(csv_save_name)
                with open(csv_save_name, 'a', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=['image_name', 'image_id', 'class_name', "country_code", 'count', 'category_id'], delimiter=';')

                    # Escreve cabeçalho apenas se o arquivo está vazio (opcional, caso queira cabeçalho)
                    if f.tell() == 0:
                        writer.writeheader()

                    writer.writerow({
                        'image_name': f"{unique_id}{ext}",
                        'image_id': unique_id,
                        'class_name': class_name, 
                        'country_code': country_code, 
                        'count': count, 
                        'category_id': category_id

                    })
                print(f"Imagem {original_filename} save.")
                print(f"Tempo da requisição: {end_time - start_time:.2f} segundos")
                print(f"Nome do arquivo:{full_local_path}")
            else:
                print(f"Something wrong with {original_filename}.")
    